RQ1 — microsaccade extraction and analysis
===========================================
Symmetry Threshold task. Previous analysis attempt wasn't good - I misunderstood -Pre-extracted saccades are essentially zero because participants are fixating, so we run a proper microsaccadedetector on the raw 1000-Hz gaze signal.

First intro in Meso et al. 2016, had to return to the og paper

Detector — Engbert & Kliegl (2003), monocular variant
-----------------------------------------------------
1.  Compute velocity with a 5-sample moving window:
        vx(n) = (x(n+2) + x(n+1) - x(n-1) - x(n-2)) / (6 * dt)
    in degrees per second (pixels are converted with
    S.ParticipantAndTaskInfo.TaskSetUpInfo.ScreenDegreesPerPixel).

2.  Per-trial median-based standard deviation of velocity:
        sigma_x = sqrt( median(vx**2) - median(vx)**2 )
    (the standard E&K robust estimator).

3.  Detection threshold:
        eta = LAMBDA * sigma         (LAMBDA = 6 by default)

4.  An event is a run of consecutive samples for which
        (vx/eta_x)**2 + (vy/eta_y)**2 > 1
    lasting at least MIN_DURATION_MS.  Successive events less than
    MIN_SEPARATION_MS apart are merged.

5.  Microsaccade filter: amplitude < AMP_MAX_DEG  (default 1.0 deg).

Per-trial we record:
    n_ms        number of microsaccades
    rate_hz     n_ms / trial duration in seconds
    mean_amp    mean amplitude (deg)
    mean_pkvel  mean peak velocity (deg/s)

We aggregate at the (pid, group, level) level over vertical-axis trials
(col 0 == VERTICAL_CODE).

For reference outputs to /Users/matyldakornacka/Desktop/DATA_DISS/RQ1/microsaccades/
    RQ1ms_per_trial.csv
    RQ1ms_per_cell.csv             (pid x level)
    RQ1ms_per_participant.csv      (pid pooled across levels)
    RQ1ms_fig1_rate_amp.png/.pdf   bars: rate & amplitude per group
    RQ1ms_fig2_by_level.png/.pdf   rate & amplitude across the 5 levels

In [1]:

import os, glob, warnings
import numpy as np
import pandas as pd
from scipy.io import loadmat
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


In [ ]:

# config
DATA_DIR = '/Users/matyldakornacka/Desktop/DATA_DISS/AllData'
OUT_DIR  = '/Users/matyldakornacka/Desktop/DATA_DISS/RQ1/microsaccades'
os.makedirs(OUT_DIR, exist_ok=True)

SCR_W, SCR_H   = 1920, 1080
SAMPLE_HZ      = 1000.0
# Wedge half-angle for the horizontal/vertical microsaccade classifier.
# A microsaccade is HORIZONTAL if |angle| < WEDGE_DEG  (rightward) or
#                                |angle| > 180-WEDGE_DEG (leftward);
# otherwise it is VERTICAL (i.e. up/down dominant).
WEDGE_DEG      = 45.0
# we now run on ALL SymThrsh trials (180 per pid) to capture both
# stimulus axes; the original vertical-only filter is retained as a
# convenience constant so other scripts can still slice if needed.
VERTICAL_CODE  = 1

# Engbert & Kliegl params
# Defaults are the *typical* monocular settings used in Engbert & Mergenthaler
# (2006) and subsequent work at 1 kHz: lambda=5, min_dur=3 ms, amp 0.03°-1°.
# Earlier we used (lambda=6, min_dur=6 ms, amp 0.05°-1°) which proved overly
# conservative and yielded only 0.46 Hz across vertical-axis trials.
LAMBDA            = 5.0
MIN_DURATION_MS   = 3
MIN_SEPARATION_MS = 12
AMP_MAX_DEG       = 1.0
AMP_MIN_DEG       = 0.03

ART_COL  = '#3D5A80'
CTRL_COL = '#EE6C4D'

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 110, 'savefig.dpi': 240, 'savefig.bbox': 'tight',
})


In [ ]:


#loading and parsing data
def load_all():
    recs, seen = [], set()
    for f in sorted(glob.glob(os.path.join(DATA_DIR, '*_AesSymPer_*.mat'))):
        try:
            S = loadmat(f, squeeze_me=True, struct_as_record=False)['S']
        except Exception:
            continue
        pid = str(S.ParticipantAndTaskInfo.PartID).strip()
        if pid in seen: continue
        seen.add(pid)
        grp = 'Artist' if (pid[0] == 'A' or pid == 'S11') else 'Control'
        dpp = float(S.ParticipantAndTaskInfo.TaskSetUpInfo.ScreenDegreesPerPixel)
        recs.append(dict(
            pid=pid, group=grp, dpp=dpp,
            ResFile=S.SymmetryThresholdTask.Data.ResFile,
            SymThrsh=S.EyeMovementForAllTasks.SymThrsh))
    return recs


def clean_xy(X, Y):
    X = np.asarray(X, dtype=float); Y = np.asarray(Y, dtype=float)
    bad = (~np.isfinite(X)) | (~np.isfinite(Y)) | \
          (X < 0) | (X > SCR_W) | (Y < 0) | (Y > SCR_H)
    X = X.copy(); Y = Y.copy()
    X[bad] = np.nan; Y[bad] = np.nan
    return X, Y


In [ ]:


# detector
def _vel_5pt(x, dt):
    """5-sample moving-window velocity (E&K eq. 1).  Returns an array
    the same length as x, with edges padded by NaN."""
    v = np.full_like(x, np.nan, dtype=float)
    if len(x) < 5: return v
    v[2:-2] = (x[4:] + x[3:-1] - x[1:-3] - x[:-4]) / (6 * dt)
    return v


def detect_microsaccades(X_px, Y_px, dpp, dt=1.0 / SAMPLE_HZ):
    """Returns a list of dicts, one per detected microsaccade.
    X_px, Y_px in pixels; dpp = degrees per pixel; dt in seconds.
    Velocities and amplitudes returned in degrees and deg/s."""
    # convert to degrees
    X = X_px * dpp
    Y = Y_px * dpp
    vx = _vel_5pt(X, dt)
    vy = _vel_5pt(Y, dt)
    valid = np.isfinite(vx) & np.isfinite(vy)
    if valid.sum() < 20: return []

    # robust SD via medians (E&K)
    msx = np.median(vx[valid] ** 2) - np.median(vx[valid]) ** 2
    msy = np.median(vy[valid] ** 2) - np.median(vy[valid]) ** 2
    if msx <= 0 or msy <= 0: return []
    eta_x = LAMBDA * np.sqrt(msx)
    eta_y = LAMBDA * np.sqrt(msy)

    # detection criterion
    cond = ((vx / eta_x) ** 2 + (vy / eta_y) ** 2) > 1
    cond &= valid

    # group runs of True
    events = []
    start = None
    min_dur_samples = max(2, int(MIN_DURATION_MS * SAMPLE_HZ / 1000))
    for i, c in enumerate(cond):
        if c and start is None: start = i
        elif (not c) and start is not None:
            if i - start >= min_dur_samples:
                events.append((start, i - 1))
            start = None
    if start is not None and len(cond) - start >= min_dur_samples:
        events.append((start, len(cond) - 1))

    # merge events closer than MIN_SEPARATION_MS
    sep = max(1, int(MIN_SEPARATION_MS * SAMPLE_HZ / 1000))
    merged = []
    for s, e in events:
        if merged and s - merged[-1][1] <= sep:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append((s, e))

    out = []
    for s, e in merged:
        if e <= s: continue
        dx = X[e] - X[s];  dy = Y[e] - Y[s]
        amp = float(np.hypot(dx, dy))
        if amp < AMP_MIN_DEG or amp > AMP_MAX_DEG:
            continue
        speed = np.hypot(vx[s:e + 1], vy[s:e + 1])
        pkvel = float(np.nanmax(speed)) if len(speed) else np.nan
        ang = float(np.degrees(np.arctan2(dy, dx)))
        out.append(dict(start=s, end=e, dur_ms=(e - s + 1),
                        amp_deg=amp, pkvel_deg_s=pkvel, angle_deg=ang))
    return out


def is_horizontal(angle_deg):
    """True if the microsaccade direction is in the horizontal wedge."""
    a = abs(((angle_deg + 180) % 360) - 180)        # fold to 0..180
    return (a < WEDGE_DEG) or (a > 180 - WEDGE_DEG)


In [ ]:


#  per trial
def build_per_trial(recs):
    """One row per trial. We now keep ALL trials (both stimulus axes) and
    split events into horizontal vs vertical microsaccades."""
    rows = []
    for r in recs:
        rf = r['ResFile']
        for ti in range(len(rf)):
            stim_axis = 'Vertical' if int(rf[ti, 0]) == VERTICAL_CODE \
                                   else 'Horizontal'
            level = int(rf[ti, 1])
            tr = r['SymThrsh'].TrialLevelData[ti]
            X, Y = clean_xy(tr.XX, tr.YY)
            ev = detect_microsaccades(X, Y, r['dpp'])
            n = len(tr.XX)
            dur_s = n / SAMPLE_HZ if n > 0 else np.nan

            n_h = sum(1 for e in ev if is_horizontal(e['angle_deg']))
            n_v = len(ev) - n_h
            if ev:
                amp_all = np.mean([e['amp_deg']     for e in ev])
                pv_all  = np.mean([e['pkvel_deg_s'] for e in ev])
                amp_h = (np.mean([e['amp_deg'] for e in ev
                                  if is_horizontal(e['angle_deg'])])
                         if n_h else np.nan)
                amp_v = (np.mean([e['amp_deg'] for e in ev
                                  if not is_horizontal(e['angle_deg'])])
                         if n_v else np.nan)
            else:
                amp_all = pv_all = amp_h = amp_v = np.nan

            rows.append(dict(
                pid=r['pid'], group=r['group'],
                trial=ti, level=level, stim_axis=stim_axis,
                duration_s=dur_s,
                n_ms=len(ev), n_ms_h=n_h, n_ms_v=n_v,
                rate_hz   =(len(ev) / dur_s) if dur_s else np.nan,
                rate_hz_h =(n_h     / dur_s) if dur_s else np.nan,
                rate_hz_v =(n_v     / dur_s) if dur_s else np.nan,
                mean_amp_deg=amp_all,
                mean_amp_h_deg=amp_h, mean_amp_v_deg=amp_v,
                mean_pkvel_deg_s=pv_all))
    return pd.DataFrame(rows)


In [ ]:


#  aggregation of per-trial data
def aggregate(per_trial):
    """Per-cell (pid x level) and per-participant aggregations.
    Rates use total time, so they're directly comparable across cells."""
    g_cell = ['pid', 'group', 'level']
    g_pp   = ['pid', 'group']

    def _agg(by):
        d = (per_trial.groupby(by)
                      .agg(n_trials=('trial', 'count'),
                           n_ms=('n_ms', 'sum'),
                           n_ms_h=('n_ms_h', 'sum'),
                           n_ms_v=('n_ms_v', 'sum'),
                           dur=('duration_s', 'sum'),
                           mean_amp=('mean_amp_deg', 'mean'),
                           mean_amp_h=('mean_amp_h_deg', 'mean'),
                           mean_amp_v=('mean_amp_v_deg', 'mean'),
                           mean_pkvel=('mean_pkvel_deg_s', 'mean'))
                      .reset_index())
        d['rate_hz']   = d.n_ms   / d.dur
        d['rate_hz_h'] = d.n_ms_h / d.dur
        d['rate_hz_v'] = d.n_ms_v / d.dur
        d['prop_h']    = d.n_ms_h / d.n_ms.replace(0, np.nan)
        return d

    return _agg(g_cell), _agg(g_pp)


In [ ]:


#  figures
def _bar_with_dots(ax, vals_a, vals_c, ylabel, title):
    rng = np.random.default_rng(0)
    for xi, (vals, col) in enumerate([(vals_a, ART_COL), (vals_c, CTRL_COL)]):
        m  = np.nanmean(vals); sd = np.nanstd(vals, ddof=1)
        sem = sd / np.sqrt(len(vals))
        ax.bar(xi, m, width=0.6, color=col, alpha=0.55,
               edgecolor=col, linewidth=1.4)
        ax.errorbar(xi, m, yerr=sem, fmt='none',
                    ecolor=col, elinewidth=1.6, capsize=6)
        jit = rng.uniform(-0.12, 0.12, len(vals))
        ax.scatter(xi + jit, vals, s=42, color=col,
                   edgecolor='white', linewidth=0.8, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Artist', 'Control'])
    ax.set_xlim(-0.7, 1.7)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.25, lw=0.6); ax.set_axisbelow(True)


def fig1_rate_amp(pp):
    """4 panels:
       (1) Total rate per group  (2) Horizontal vs Vertical rate per group
       (3) Mean amplitude        (4) Proportion horizontal."""
    fig, axes = plt.subplots(1, 4, figsize=(17, 5.0),
                             gridspec_kw={'wspace': 0.42})
    a = pp[pp.group == 'Artist'];  c = pp[pp.group == 'Control']

    _bar_with_dots(axes[0], a.rate_hz.values, c.rate_hz.values,
                   'Total rate  (Hz)', 'All microsaccades')

    # grouped bars: H vs V per group
    ax = axes[1]
    rng = np.random.default_rng(1)
    pos = [0, 0.7, 2, 2.7]
    vals = [a.rate_hz_h.values, a.rate_hz_v.values,
            c.rate_hz_h.values, c.rate_hz_v.values]
    cols = [ART_COL, ART_COL, CTRL_COL, CTRL_COL]
    hatches = ['', '//', '', '//']
    for p, v, col, h in zip(pos, vals, cols, hatches):
        m = np.nanmean(v); sem = np.nanstd(v, ddof=1) / np.sqrt(len(v))
        ax.bar(p, m, width=0.55, color=col, alpha=0.55,
               edgecolor=col, linewidth=1.4, hatch=h)
        ax.errorbar(p, m, yerr=sem, fmt='none', ecolor=col,
                    elinewidth=1.4, capsize=4)
        ax.scatter(p + rng.uniform(-0.1, 0.1, len(v)), v,
                   s=24, color=col, edgecolor='white', lw=0.6, zorder=3)
    ax.set_xticks([0.35, 2.35]); ax.set_xticklabels(['Artist', 'Control'])
    ax.set_ylabel('Rate  (Hz)')
    ax.set_title('Horizontal vs Vertical')
    ax.grid(axis='y', alpha=0.25, lw=0.6); ax.set_axisbelow(True)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor='0.6', edgecolor='0.4', label='Horizontal'),
                       Patch(facecolor='0.6', edgecolor='0.4', hatch='//',
                             label='Vertical')],
              frameon=False, fontsize=9, loc='upper right')

    _bar_with_dots(axes[2], a.mean_amp.values, c.mean_amp.values,
                   'Mean amplitude  (°)', 'Amplitude (all)')
    _bar_with_dots(axes[3], a.prop_h.values, c.prop_h.values,
                   'Proportion horizontal', 'Direction balance')
    axes[3].axhline(0.5, color='0.4', ls='--', lw=0.8)

    fig.suptitle('Microsaccades during the Symmetry Threshold task  '
                 '(all trials, both stimulus axes)',
                 fontsize=13, fontweight='bold', y=1.04)
    fig.text(0.5, -0.04,
             'Microsaccades classified as Horizontal if |angle| < 45° or '
             '> 135°; Vertical otherwise.   Bars = group mean ± 1 SEM, '
             'dots = participants.',
             ha='center', fontsize=9, style='italic', color='#555')
    out = os.path.join(OUT_DIR, 'RQ1ms_fig1_rate_amp')
    fig.savefig(out + '.png'); fig.savefig(out + '.pdf')
    plt.close(fig)
    print('  wrote', out + '.png')


def fig2_by_level(cell):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2),
                             gridspec_kw={'wspace': 0.28})
    levels = [1, 2, 3, 4, 5]
    for ax, metric, ylabel, title in [
            (axes[0], 'rate_hz',  'Microsaccade rate  (Hz)',
             'Rate across luminance levels'),
            (axes[1], 'mean_amp', 'Mean amplitude  (°)',
             'Amplitude across luminance levels')]:
        for grp, col in [('Artist',  ART_COL), ('Control', CTRL_COL)]:
            sub = cell[cell.group == grp]
            mean = sub.groupby('level')[metric].mean().reindex(levels)
            sem  = (sub.groupby('level')[metric].std(ddof=1)
                       / np.sqrt(sub.groupby('level').size())) \
                   .reindex(levels)
            ax.errorbar(levels, mean, yerr=sem, marker='o',
                        color=col, lw=1.6, capsize=4,
                        label=grp, markersize=7)
        ax.set_xticks(levels); ax.set_xlabel('Luminance level')
        ax.set_ylabel(ylabel); ax.set_title(title)
        ax.grid(alpha=0.25, lw=0.6); ax.set_axisbelow(True)
        ax.legend(frameon=False, loc='best')
    fig.suptitle('Microsaccades across the 5 luminance levels',
                 fontsize=13, fontweight='bold', y=1.02)
    out = os.path.join(OUT_DIR, 'RQ1ms_fig2_by_level')
    fig.savefig(out + '.png'); fig.savefig(out + '.pdf')
    plt.close(fig)
    print('  wrote', out + '.png')


In [ ]:


#  main logic and stats
def run():
    recs = load_all()
    na = sum(r['group'] == 'Artist'  for r in recs)
    nc = sum(r['group'] == 'Control' for r in recs)
    print(f'Loaded {len(recs)} participants ({na} artists, {nc} controls)')
    print(f'LAMBDA={LAMBDA}, min_duration={MIN_DURATION_MS}ms, '
          f'amp range=[{AMP_MIN_DEG},{AMP_MAX_DEG}]°\n')

    print('Detecting microsaccades on every vertical-axis trial...')
    per_trial = build_per_trial(recs)
    per_trial.to_csv(os.path.join(OUT_DIR, 'RQ1ms_per_trial.csv'), index=False)
    cell, pp = aggregate(per_trial)
    cell.to_csv(os.path.join(OUT_DIR, 'RQ1ms_per_cell.csv'),       index=False)
    pp.to_csv  (os.path.join(OUT_DIR, 'RQ1ms_per_participant.csv'),index=False)

    print(f'  rows: per-trial = {len(per_trial)}, '
          f'per-cell = {len(cell)}, per-participant = {len(pp)}\n')

    from scipy import stats
    a = pp[pp.group == 'Artist'];  c = pp[pp.group == 'Control']

    print('Per-participant microsaccade summary (mean ± SD)')
    for label, key in [('Total rate (Hz)',         'rate_hz'),
                       ('Horizontal rate (Hz)',    'rate_hz_h'),
                       ('Vertical rate (Hz)',      'rate_hz_v'),
                       ('Mean amplitude (deg)',    'mean_amp'),
                       ('Proportion horizontal',   'prop_h')]:
        ma, sa = a[key].mean(), a[key].std(ddof=1)
        mc, sc = c[key].mean(), c[key].std(ddof=1)
        t, p = stats.ttest_ind(a[key], c[key], equal_var=False,
                               nan_policy='omit')
        print(f'  {label:<24}  A={ma:+.3f}±{sa:.3f}   '
              f'C={mc:+.3f}±{sc:.3f}   t={t:+.2f}  p={p:.4f}')

    print('\nPlotting...')
    fig1_rate_amp(pp)
    fig2_by_level(cell)

    print('\nDone. Outputs in:', OUT_DIR)
    for f in sorted(os.listdir(OUT_DIR)):
        print(' ', f)


if __name__ == '__main__':
    run()


Loaded 31 participants (13 artists, 18 controls)
LAMBDA=5.0, min_duration=3ms, amp range=[0.03,1.0]°

Detecting microsaccades on every vertical-axis trial...
  rows: per-trial = 5580, per-cell = 155, per-participant = 31

=== Per-participant microsaccade summary (mean ± SD) ===
  Total rate (Hz)           A=+1.366±0.415   C=+1.710±0.534   t=-2.01  p=0.0536
  Horizontal rate (Hz)      A=+0.756±0.334   C=+0.931±0.306   t=-1.50  p=0.1474
  Vertical rate (Hz)        A=+0.610±0.181   C=+0.779±0.321   t=-1.85  p=0.0747
  Mean amplitude (deg)      A=+0.226±0.100   C=+0.222±0.097   t=+0.09  p=0.9259
  Proportion horizontal     A=+0.539±0.114   C=+0.546±0.094   t=-0.20  p=0.8460

Plotting...
  wrote /Users/matyldakornacka/Desktop/DATA_DISS/RQ1/microsaccades/RQ1ms_fig1_rate_amp.png
  wrote /Users/matyldakornacka/Desktop/DATA_DISS/RQ1/microsaccades/RQ1ms_fig2_by_level.png

Done. Outputs in: /Users/matyldakornacka/Desktop/DATA_DISS/RQ1/microsaccades
  RQ1ms_anova_amplitude.csv
  RQ1ms_anova_rate.c